## Objectifs

1. lire les événements de perturbation depuis Kafka ;
2. calculer deux indicateurs temps réel à l’aide de fenêtres temporelles ;
3. enrichir les résultats avec des données statiques provenant de Garage ;
4. préparer l’alimentation du dashboard déjà développé dans le projet.

### 1. Initialisation de l’environnement Spark

In [12]:
import os
import json

from pyspark import SparkContext, SparkConf
from pyspark.sql import SQLContext
from pyspark.sql import SparkSession

from pyspark.sql.functions import (
    from_json, col, window, count, to_timestamp, lower, trim
)
from pyspark.sql.types import (
    StructType, StructField, StringType, TimestampType
)

In [45]:
conf = SparkConf() \
    .setAppName("IDFM_Streaming_Project") \
    .setMaster("spark://spark:7077") \
    .set(
        "spark.jars.packages",
        "org.apache.hadoop:hadoop-aws:3.3.4,"
        "com.amazonaws:aws-java-sdk-bundle:1.12.262,"
        "org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.3,"
        "org.apache.spark:spark-hadoop-cloud_2.12:3.5.3"
    ) \
    .set("spark.sql.shuffle.partitions", "10") \
    .set("spark.hadoop.fs.s3a.committer.name", "staging") \
    .set("spark.hadoop.mapreduce.outputcommitter.factory.scheme.s3a", "org.apache.hadoop.fs.s3a.commit.S3ACommitterFactory") \
    .set("spark.hadoop.fs.s3a.committer.staging.tmp.path", "/tmp/s3a-commit") \
    .set("spark.hadoop.fs.s3a.committer.staging.unique-filenames", "true") \
    .set("spark.hadoop.fs.s3a.committer.staging.conflict-mode", "replace")

sc = SparkContext.getOrCreate(conf=conf)
sql_context = SQLContext(sc)
spark = SparkSession(sc)
spark.sparkContext.setLogLevel("WARN")

/opt/conda/lib/python3.12/site-packages/pyspark/sql/context.py:113: FutureWarning: Deprecated in 3.0.0. Use SparkSession.builder.getOrCreate() instead.
  warnings.warn(


### 2. Configuration des sources de données

In [46]:
# Kafka
kafka_broker = "kafka1:9092"
kafka_topic = "disruptions"

# Variables d'environnement Garage / S3A
bucket_name = os.getenv("bucket_name", "your-bucket-name")
access_key = os.getenv("key_id")
secret_key = os.getenv("secret_key")
garage_endpoint = os.getenv("AWS_S3_ENDPOINT", "http://garage:3900")

In [47]:
sc._jsc.hadoopConfiguration().set("fs.s3a.endpoint", f"http://{os.getenv('minio_ip_address')}:3900")
sc._jsc.hadoopConfiguration().set("fs.s3a.access.key", os.getenv("key_id"))
sc._jsc.hadoopConfiguration().set("fs.s3a.secret.key", os.getenv("secret_key"))
sc._jsc.hadoopConfiguration().set("fs.s3a.endpoint.region", "garage")
sc._jsc.hadoopConfiguration().set("fs.s3a.path.style.access", "true")
sc._jsc.hadoopConfiguration().set("fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
sc._jsc.hadoopConfiguration().set("fs.s3a.connection.ssl.enabled", "false")

### 3. Définition du schéma des messages de perturbation

L’inspection des messages Kafka montre que chaque message du topic `disruptions` contient une **liste JSON de perturbations** et non un unique objet.

Nous devons donc :
1. parser le message comme un tableau (`ArrayType`) ;
2. exploser ce tableau pour obtenir une ligne par perturbation ;
3. extraire ensuite les champs utiles pour l’analyse streaming.

In [48]:
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType, ArrayType
)

In [49]:
severity_schema = StructType([
    StructField("name", StringType(), True),
    StructField("effect", StringType(), True),
    StructField("color", StringType(), True),
    StructField("priority", IntegerType(), True)
])

line_schema = StructType([
    StructField("id", StringType(), True),
    StructField("name", StringType(), True),
    StructField("code", StringType(), True)
])

pt_object_schema = StructType([
    StructField("id", StringType(), True),
    StructField("embedded_type", StringType(), True),
    StructField("line", line_schema, True)
])

impacted_object_schema = StructType([
    StructField("pt_object", pt_object_schema, True)
])

disruption_schema = StructType([
    StructField("id", StringType(), True),
    StructField("disruption_id", StringType(), True),
    StructField("status", StringType(), True),
    StructField("updated_at", StringType(), True),
    StructField("cause", StringType(), True),
    StructField("category", StringType(), True),
    StructField("severity", severity_schema, True),
    StructField("impacted_objects", ArrayType(impacted_object_schema), True)
])

disruptions_array_schema = ArrayType(disruption_schema)

### 4. Parsing du flux Kafka

Chaque message Kafka est converti depuis JSON vers un tableau de perturbations.  
Nous utilisons ensuite `explode` pour transformer ce tableau en une table où chaque ligne correspond à une perturbation.

In [50]:
from pyspark.sql.functions import from_json, col, explode, to_timestamp

In [51]:
raw_stream = sql_context.readStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers", kafka_broker) \
    .option("subscribe", kafka_topic) \
    .option("startingOffsets", "latest") \
    .load()

In [52]:
parsed_disruptions = raw_stream \
    .selectExpr("CAST(value AS STRING) AS message") \
    .select(from_json(col("message"), disruptions_array_schema).alias("disruptions")) \
    .select(explode(col("disruptions")).alias("disruption"))

In [53]:
events_stream = parsed_disruptions.select(
    col("disruption.id").alias("event_id"),
    col("disruption.disruption_id").alias("disruption_id"),
    col("disruption.status").alias("status"),
    col("disruption.cause").alias("cause"),
    col("disruption.category").alias("category"),
    col("disruption.severity.name").alias("severity_name"),
    col("disruption.severity.effect").alias("severity_effect"),
    col("disruption.severity.priority").alias("severity_priority"),
    to_timestamp(col("disruption.updated_at"), "yyyyMMdd'T'HHmmss").alias("event_time"),
    col("disruption.impacted_objects").alias("impacted_objects")
).filter(
    col("event_time").isNotNull()
).dropDuplicates(["event_id"]).withWatermark("event_time", "10 minutes")

### 5. Requête 1 : évolution des perturbations par niveau de sévérité

Cette requête permet d’observer la distribution des perturbations selon leur niveau de gravité au cours du temps.

Les données sont agrégées dans des fenêtres temporelles afin de lisser les variations et faciliter la lecture dans un dashboard.

In [54]:
from pyspark.sql.functions import window, count, col

In [55]:
disruptions_by_severity = events_stream \
    .filter(col("severity_name").isNotNull()) \
    .groupBy(
        window(col("event_time"), "15 minutes", "5 minutes"),
        col("severity_name")
    ) \
    .agg(count("*").alias("nb_events")) \
    .select(
        col("window.start").alias("window_start"),
        col("window.end").alias("window_end"),
        col("severity_name"),
        col("nb_events")
    )

In [56]:
query_severity = disruptions_by_severity.writeStream \
    .outputMode("update") \
    .format("console") \
    .option("truncate", False) \
    .option("checkpointLocation", "/tmp/checkpoints/...") \
    .start()

26/03/27 05:36:54 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


26/03/27 05:36:59 WARN AdminClientConfig: These configurations '[key.deserializer, value.deserializer, enable.auto.commit, max.poll.records, auto.offset.reset]' were supplied but are not used yet.


-------------------------------------------
Batch: 13
-------------------------------------------
+-------------------+-------------------+-------------+---------+
|window_start       |window_end         |severity_name|nb_events|
+-------------------+-------------------+-------------+---------+
|2026-03-27 06:15:00|2026-03-27 06:30:00|perturbée    |3        |
|2026-03-27 06:05:00|2026-03-27 06:20:00|perturbée    |9        |
|2026-03-27 06:10:00|2026-03-27 06:25:00|perturbée    |6        |
+-------------------+-------------------+-------------+---------+



-------------------------------------------
Batch: 14
-------------------------------------------
+------------+----------+-------------+---------+
|window_start|window_end|severity_name|nb_events|
+------------+----------+-------------+---------+
+------------+----------+-------------+---------+



-------------------------------------------
Batch: 15
-------------------------------------------
+------------+----------+-------------+---------+
|window_start|window_end|severity_name|nb_events|
+------------+----------+-------------+---------+
+------------+----------+-------------+---------+



-------------------------------------------
Batch: 16
-------------------------------------------
+-------------------+-------------------+-------------+---------+
|window_start       |window_end         |severity_name|nb_events|
+-------------------+-------------------+-------------+---------+
|2026-03-27 06:15:00|2026-03-27 06:30:00|perturbée    |4        |
|2026-03-27 06:20:00|2026-03-27 06:35:00|perturbée    |1        |
|2026-03-27 06:10:00|2026-03-27 06:25:00|perturbée    |7        |
+-------------------+-------------------+-------------+---------+



-------------------------------------------
Batch: 17
-------------------------------------------
+-------------------+-------------------+------------------+---------+
|window_start       |window_end         |severity_name     |nb_events|
+-------------------+-------------------+------------------+---------+
|2026-03-27 06:20:00|2026-03-27 06:35:00|Bloquante passante|7        |
|2026-03-27 06:35:00|2026-03-27 06:50:00|perturbée         |1        |
|2026-03-27 06:10:00|2026-03-27 06:25:00|Bloquante passante|2        |
|2026-03-27 06:15:00|2026-03-27 06:30:00|Bloquante passante|2        |
|2026-03-27 06:30:00|2026-03-27 06:45:00|perturbée         |2        |
|2026-03-27 06:25:00|2026-03-27 06:40:00|Bloquante passante|6        |
|2026-03-27 06:25:00|2026-03-27 06:40:00|perturbée         |2        |
|2026-03-27 06:30:00|2026-03-27 06:45:00|Bloquante passante|5        |
|2026-03-27 06:20:00|2026-03-27 06:35:00|perturbée         |2        |
+-------------------+-------------------+---------

-------------------------------------------
Batch: 18
-------------------------------------------
+-------------------+-------------------+-------------+---------+
|window_start       |window_end         |severity_name|nb_events|
+-------------------+-------------------+-------------+---------+
|2026-03-27 06:15:00|2026-03-27 06:30:00|perturbée    |10       |
|2026-03-27 06:05:00|2026-03-27 06:20:00|perturbée    |11       |
|2026-03-27 06:25:00|2026-03-27 06:40:00|perturbée    |4        |
|2026-03-27 06:20:00|2026-03-27 06:35:00|perturbée    |6        |
|2026-03-27 06:10:00|2026-03-27 06:25:00|perturbée    |11       |
+-------------------+-------------------+-------------+---------+



In [57]:
query_severity.stop()

### 6. Requête 2 : perturbations par ligne

Cette requête permet d’identifier les lignes de transport les plus impactées par des perturbations.

Les lignes étant imbriquées dans le champ `impacted_objects`, une transformation supplémentaire est nécessaire pour extraire les objets de type `line`.

In [58]:
from pyspark.sql.functions import explode

- Collection :

In [59]:
line_events_stream = events_stream \
    .select(
        "event_id",
        "event_time",
        explode(col("impacted_objects")).alias("impacted_object")
    ) \
    .select(
        col("event_id"),
        col("event_time"),
        col("impacted_object.pt_object.embedded_type").alias("embedded_type"),
        col("impacted_object.pt_object.line.id").alias("line_id"),
        col("impacted_object.pt_object.line.name").alias("line_name"),
        col("impacted_object.pt_object.line.code").alias("line_code")
    ) \
    .filter(col("embedded_type") == "line") \
    .filter(col("line_id").isNotNull()) \
    .dropDuplicates(["event_id", "line_id"])

- Aggrégation :

In [60]:
disruptions_by_line = line_events_stream \
    .groupBy(
        window(col("event_time"), "10 minutes", "2 minutes"),
        col("line_id"),
        col("line_name"),
        col("line_code")
    ) \
    .agg(count("*").alias("nb_disruptions")) \
    .select(
        col("window.start").alias("window_start"),
        col("window.end").alias("window_end"),
        col("line_id"),
        col("line_name"),
        col("line_code"),
        col("nb_disruptions")
    )

In [ ]:
query_line = disruptions_by_line.writeStream \
    .outputMode("update") \
    .format("console") \
    .option("truncate", False) \
    .option("checkpointLocation", "/tmp/checkpoints/line") \
    .start()

26/03/27 05:37:43 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


26/03/27 05:37:43 WARN AdminClientConfig: These configurations '[key.deserializer, value.deserializer, enable.auto.commit, max.poll.records, auto.offset.reset]' were supplied but are not used yet.


-------------------------------------------
Batch: 5
-------------------------------------------
+-------------------+-------------------+----------------+---------+---------+--------------+
|window_start       |window_end         |line_id         |line_name|line_code|nb_disruptions|
+-------------------+-------------------+----------------+---------+---------+--------------+
|2026-03-27 06:32:00|2026-03-27 06:42:00|line:IDFM:C01176|152      |152      |3             |
|2026-03-27 06:20:00|2026-03-27 06:30:00|line:IDFM:C01075|24       |24       |1             |
|2026-03-27 06:24:00|2026-03-27 06:34:00|line:IDFM:C01379|9        |9        |1             |
|2026-03-27 06:22:00|2026-03-27 06:32:00|line:IDFM:C01379|9        |9        |1             |
|2026-03-27 06:28:00|2026-03-27 06:38:00|line:IDFM:C01111|76       |76       |1             |
|2026-03-27 06:20:00|2026-03-27 06:30:00|line:IDFM:C00550|3121     |3121     |1             |
|2026-03-27 06:06:00|2026-03-27 06:16:00|line:IDFM:C01532

In [62]:
query_line.stop()

### 7. Jointure avec des données statiques stockées dans Garage

Afin d’enrichir les résultats du streaming, nous utilisons un référentiel statique des lignes stocké dans Garage.

Cette jointure permet d’associer aux identifiants techniques des lignes des informations complémentaires utiles à l’analyse et à l’affichage dans le dashboard, comme le mode de transport, le réseau ou d’autres métadonnées.

In [64]:
bucket_name = os.getenv("bucket_name", "graphxproject")

lines_df = spark.read \
    .option("header", True) \
    .option("inferSchema", True) \
    .csv(f"s3a://{bucket_name}/data/lines_links.csv")

lines_df.printSchema()
lines_df.show(5, truncate=False)

26/03/27 05:38:11 WARN FileStreamSink: Assume no metadata directory. Error while looking for metadata directory in the path: s3a://graphxproject/data/lines_links.csv.
org.apache.hadoop.fs.s3a.AWSBadRequestException: getFileStatus on s3a://graphxproject/data/lines_links.csv: com.amazonaws.services.s3.model.AmazonS3Exception: Authorization header malformed, unexpected scope: 20260327/us-east-1/s3/aws4_request (Service: Amazon S3; Status Code: 400; Error Code: AuthorizationHeaderMalformed; Request ID: null; S3 Extended Request ID: null; Proxy: null), S3 Extended Request ID: null:AuthorizationHeaderMalformed: Authorization header malformed, unexpected scope: 20260327/us-east-1/s3/aws4_request (Service: Amazon S3; Status Code: 400; Error Code: AuthorizationHeaderMalformed; Request ID: null; S3 Extended Request ID: null; Proxy: null)
	at org.apache.hadoop.fs.s3a.S3AUtils.translateException(S3AUtils.java:249)
	at org.apache.hadoop.fs.s3a.S3AUtils.translateException(S3AUtils.java:175)
	at org.

Py4JJavaError: An error occurred while calling o579.csv.
: org.apache.hadoop.fs.s3a.AWSBadRequestException: getFileStatus on s3a://graphxproject/data/lines_links.csv: com.amazonaws.services.s3.model.AmazonS3Exception: Bad Request (Service: Amazon S3; Status Code: 400; Error Code: 400 Bad Request; Request ID: null; S3 Extended Request ID: null; Proxy: null), S3 Extended Request ID: null:400 Bad Request: Bad Request (Service: Amazon S3; Status Code: 400; Error Code: 400 Bad Request; Request ID: null; S3 Extended Request ID: null; Proxy: null)
	at org.apache.hadoop.fs.s3a.S3AUtils.translateException(S3AUtils.java:249)
	at org.apache.hadoop.fs.s3a.S3AUtils.translateException(S3AUtils.java:175)
	at org.apache.hadoop.fs.s3a.S3AFileSystem.s3GetFileStatus(S3AFileSystem.java:3796)
	at org.apache.hadoop.fs.s3a.S3AFileSystem.innerGetFileStatus(S3AFileSystem.java:3688)
	at org.apache.hadoop.fs.s3a.S3AFileSystem.lambda$exists$34(S3AFileSystem.java:4703)
	at org.apache.hadoop.fs.statistics.impl.IOSt

In [66]:
import pandas as pd

lines_url = "https://data.iledefrance-mobilites.fr/api/explore/v2.1/catalog/datasets/traces-du-reseau-ferre-idf/exports/csv"

lines_pd = pd.read_csv(lines_url, sep=";")
lines_pd.head()

,geo_point_2d,geo_shape,objectid_1,idrefliga,idrefligc,res_com,reseau,mode,train,rer,...,exploitant,date_mes,idf,extcode,indice_lig,shape_leng,colourweb_hexa,colourprint_cmjn,picto,picto_final
0,"48.899924648904005, 2.5091608671432195","{""coordinates"": [[2.506408247006797, 48.897143...",10,A01761,C01843,TRAM 4,TRAMWAY,TRAMWAY,0,0,...,SNCF,2006-11-20T00:00:00+00:00,1,800:T4,4,737.124862,dfaf47,0 19 88 13,NaN,https://data.iledefrance-mobilites.fr/explore/...
1,"48.74207585644005, 2.331916074867983","{""coordinates"": [[2.312711013515798, 48.747941...",143,A01840,C01727,RER C,RER C,RER,0,1,...,SNCF,1886-10-18T00:00:00+00:00,1,800:C,C,3148.308434,ffcc30,0 19 100 0,NaN,https://data.iledefrance-mobilites.fr/explore/...
2,"48.53264201027134, 2.0391545291806703","{""coordinates"": [[2.008771813873437, 48.533732...",224,A01840,C01727,RER C,RER C,RER,0,1,...,SNCF,1865-12-28T00:00:00+00:00,1,800:C,C,4574.235763,ffcc30,0 19 100 0,NaN,https://data.iledefrance-mobilites.fr/explore/...
3,"48.556641907342936, 2.1479735537006155","{""coordinates"": [[2.124937383537765, 48.550926...",225,A01840,C01727,RER C,RER C,RER,0,1,...,SNCF,1865-12-28T00:00:00+00:00,1,800:C,C,3692.928336,ffcc30,0 19 100 0,NaN,https://data.iledefrance-mobilites.fr/explore/...
4,"48.59550184143638, 2.282097752647657","{""coordinates"": [[2.266213853170556, 48.591119...",227,A01840,C01727,RER C,RER C,RER,0,1,...,SNCF,1865-12-28T00:00:00+00:00,1,800:C,C,2613.891228,ffcc30,0 19 100 0,NaN,https://data.iledefrance-mobilites.fr/explore/...


In [67]:
lines_df = spark.createDataFrame(lines_pd)
print(lines_df.columns)
lines_df.show(5, truncate=False)
print(lines_df.columns)

['geo_point_2d', 'geo_shape', 'objectid_1', 'idrefliga', 'idrefligc', 'res_com', 'reseau', 'mode', 'train', 'rer', 'metro', 'tramway', 'val', 'exploitant', 'date_mes', 'idf', 'extcode', 'indice_lig', 'shape_leng', 'colourweb_hexa', 'colourprint_cmjn', 'picto', 'picto_final']


26/03/27 05:39:56 WARN TaskSetManager: Stage 26 contains a task of very large size (1320 KiB). The maximum recommended task size is 1000 KiB.


+--------------------------------------+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [69]:
from pyspark.sql.functions import col, concat, lit

lines_ref_df = lines_df.select(
    concat(lit("line:IDFM:"), col("idrefligc")).alias("ref_line_id"),
    col("indice_lig").cast("string").alias("ref_line_label"),
    col("reseau").cast("string").alias("network_name"),
    col("mode").cast("string").alias("transport_mode"),
    col("exploitant").cast("string").alias("operator_name"),
    col("colourweb_hexa").cast("string").alias("line_color")
).dropDuplicates(["ref_line_id"])

lines_ref_df.show(10, truncate=False)

26/03/27 05:41:08 WARN TaskSetManager: Stage 27 contains a task of very large size (1320 KiB). The maximum recommended task size is 1000 KiB.


+----------------+--------------+------------+--------------+-------------+----------+
|ref_line_id     |ref_line_label|network_name|transport_mode|operator_name|line_color|
+----------------+--------------+------------+--------------+-------------+----------+
|line:IDFM:C00563|CDG           |CDGVAL      |NAVETTE       |TRANSDEV     |5cc5ed    |
|line:IDFM:C01371|1             |METRO       |METRO         |RATP         |ffbe00    |
|line:IDFM:C01372|2             |METRO       |METRO         |RATP         |0055c8    |
|line:IDFM:C01373|3             |METRO       |METRO         |RATP         |6e6e00    |
|line:IDFM:C01374|4             |METRO       |METRO         |RATP         |a0006e    |
|line:IDFM:C01375|5             |METRO       |METRO         |RATP         |ff7e2e    |
|line:IDFM:C01376|6             |METRO       |METRO         |RATP         |6eca97    |
|line:IDFM:C01377|7             |METRO       |METRO         |RATP         |f49fb3    |
|line:IDFM:C01378|8             |METRO     

In [71]:
disruptions_by_line_enriched = disruptions_by_line.join(
    lines_ref_df,
    disruptions_by_line.line_id == lines_ref_df.ref_line_id,
    "left"
).select(
    col("window_start"),
    col("window_end"),
    col("line_id"),
    col("line_name"),
    col("line_code"),
    col("transport_mode"),
    col("network_name"),
    col("operator_name"),
    col("line_color"),
    col("nb_disruptions")
)

In [72]:
query_line_enriched = disruptions_by_line_enriched.writeStream \
    .outputMode("update") \
    .format("console") \
    .option("truncate", False) \
    .option("checkpointLocation", "/tmp/checkpoints/disruptions_by_line_enriched") \
    .start()

26/03/27 05:42:09 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


26/03/27 05:42:09 WARN AdminClientConfig: These configurations '[key.deserializer, value.deserializer, enable.auto.commit, max.poll.records, auto.offset.reset]' were supplied but are not used yet.
26/03/27 05:42:09 WARN TaskSetManager: Stage 30 contains a task of very large size (1320 KiB). The maximum recommended task size is 1000 KiB.


-------------------------------------------
Batch: 0
-------------------------------------------
+------------+----------+-------+---------+---------+--------------+------------+-------------+----------+--------------+
|window_start|window_end|line_id|line_name|line_code|transport_mode|network_name|operator_name|line_color|nb_disruptions|
+------------+----------+-------+---------+---------+--------------+------------+-------------+----------+--------------+
+------------+----------+-------+---------+---------+--------------+------------+-------------+----------+--------------+



26/03/27 05:42:17 WARN TaskSetManager: Stage 36 contains a task of very large size (1320 KiB). The maximum recommended task size is 1000 KiB.


-------------------------------------------
Batch: 1
-------------------------------------------
+-------------------+-------------------+----------------+---------+---------+--------------+------------+-------------+----------+--------------+
|window_start       |window_end         |line_id         |line_name|line_code|transport_mode|network_name|operator_name|line_color|nb_disruptions|
+-------------------+-------------------+----------------+---------+---------+--------------+------------+-------------+----------+--------------+
|2026-02-03 13:36:00|2026-02-03 13:46:00|line:IDFM:C01681|3        |3        |NULL          |NULL        |NULL         |NULL      |1             |
|2026-02-03 13:30:00|2026-02-03 13:40:00|line:IDFM:C01681|3        |3        |NULL          |NULL        |NULL         |NULL      |1             |
|2026-02-03 13:38:00|2026-02-03 13:48:00|line:IDFM:C01681|3        |3        |NULL          |NULL        |NULL         |NULL      |1             |
|2026-02-03 13:32:00|

26/03/27 05:42:20 WARN TaskSetManager: Stage 46 contains a task of very large size (1320 KiB). The maximum recommended task size is 1000 KiB.


-------------------------------------------
Batch: 2
-------------------------------------------
+-------------------+-------------------+----------------+---------+---------+--------------+------------+-------------+----------+--------------+
|window_start       |window_end         |line_id         |line_name|line_code|transport_mode|network_name|operator_name|line_color|nb_disruptions|
+-------------------+-------------------+----------------+---------+---------+--------------+------------+-------------+----------+--------------+
|2026-03-06 10:14:00|2026-03-06 10:24:00|line:IDFM:C00112|5314     |5314     |NULL          |NULL        |NULL         |NULL      |1             |
|2026-03-06 10:12:00|2026-03-06 10:22:00|line:IDFM:C00112|5314     |5314     |NULL          |NULL        |NULL         |NULL      |1             |
|2026-03-06 10:08:00|2026-03-06 10:18:00|line:IDFM:C00112|5314     |5314     |NULL          |NULL        |NULL         |NULL      |1             |
|2026-03-06 10:06:00|

26/03/27 05:42:23 WARN TaskSetManager: Stage 48 contains a task of very large size (1320 KiB). The maximum recommended task size is 1000 KiB.


-------------------------------------------
Batch: 3
-------------------------------------------
+-------------------+-------------------+----------------+---------+---------+--------------+------------+-------------+----------+--------------+
|window_start       |window_end         |line_id         |line_name|line_code|transport_mode|network_name|operator_name|line_color|nb_disruptions|
+-------------------+-------------------+----------------+---------+---------+--------------+------------+-------------+----------+--------------+
|2026-03-27 06:32:00|2026-03-27 06:42:00|line:IDFM:C01176|152      |152      |NULL          |NULL        |NULL         |NULL      |3             |
|2026-03-27 06:30:00|2026-03-27 06:40:00|line:IDFM:C01176|152      |152      |NULL          |NULL        |NULL         |NULL      |3             |
|2026-03-27 06:24:00|2026-03-27 06:34:00|line:IDFM:C01176|152      |152      |NULL          |NULL        |NULL         |NULL      |3             |
|2026-03-27 06:28:00|

In [ ]:
query_line_enriched.stop()